# Quantum Hodgkin–Huxley Transformer — Hybrid Architecture

### Design Principles
1. **Zero-initialized output** — QHH block output starts at exactly 0. Model begins as a pure transformer.
2. **Detached input** — QHH receives `x.detach()`, so its 128-step backward never reaches the embedding.
3. **Separate gradient clipping** — QHH and transformer gradients clipped independently, so noisy QHH gradients can't suppress clean transformer learning.
4. **Additive residual** — `x + hh_out`. Embedding `x` is never scaled.

### Experimental Design
- **QHH-Transformer** vs **Transformer baseline** (same class, `use_qhh=False`)
- Baseline weights copied to hybrid at init → identical starting point
- Any PPL improvement comes solely from the QHH block

### Modes
`MODE = 'quick' | 'serious' | 'neurips'`

In [1]:
#!pip install -q torch torchvision torchaudio
!pip install -q transformers datasets tokenizers
!pip install -q tqdm matplotlib
import subprocess
r = subprocess.run(['python','-c',
    'import torch; print(f"PyTorch {torch.__version__}"); '
    'print(f"CUDA {torch.version.cuda}"); print(torch.cuda.get_device_name(0))'],
    capture_output=True, text=True)
print(r.stdout.strip())


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
PyTorch 2.4.1+cu124
CUDA 12.4
NVIDIA H100 80GB HBM3


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import math, time, json
import numpy as np
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import GPT2TokenizerFast
from pathlib import Path

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

# ═══════════════════════════════════════════
MODE = 'quick'  # 'quick' | 'serious' | 'neurips'
# ═══════════════════════════════════════════

# max_train/val/test = number of WikiText ROWS to select (then concatenated + chunked).
# With concatenation, ~40 tokens/row → need more rows than before to get enough sequences.
MC = {
    'quick':   dict(max_train=50_000,  max_val=5_000,  max_test=5_000,
                    epochs=10, bs=64, seq=128, d=256, heads=4, layers=4, lr=1e-3),
    'serious': dict(max_train=500_000, max_val=10_000, max_test=10_000,
                    epochs=15, bs=96, seq=256, d=384, heads=6, layers=5, lr=6e-4),
    'neurips': dict(max_train=None,    max_val=None,   max_test=None,
                    epochs=25, bs=128, seq=256, d=512, heads=8, layers=6, lr=5e-4),
}[MODE]

class Config:
    vocab_size = 50257
    d_model = MC['d']; n_heads = MC['heads']; n_layers = MC['layers']
    dropout = 0.1; max_seq_len = MC['seq']

    # HH
    dt=0.5; n_substeps=2; v_thresh=-20.0; v_reset=-65.0

    # Quantum
    omega_init=1.0; epsilon_init=0.1; gamma_deph_init=0.5

    # Training
    batch_size=MC['bs']; lr=MC['lr']; weight_decay=0.05
    epochs=MC['epochs']; warmup_frac=0.05; grad_clip=1.0; patience=5
    qhh_grad_clip=5.0  # Separate, more lenient clip for QHH params

    # Data
    max_train=MC['max_train']; max_val=MC['max_val']; max_test=MC['max_test']
    workers=4; device=torch.device('cuda'); dtype=torch.bfloat16

    save_dir = Path(f'./qhh_hybrid_{MODE}')
    save_dir.mkdir(exist_ok=True, parents=True)

cfg = Config()
print(f"MODE='{MODE}' d={cfg.d_model} heads={cfg.n_heads} layers={cfg.n_layers} "
      f"seq={cfg.max_seq_len} bs={cfg.batch_size} epochs={cfg.epochs}")

MODE='quick' d=256 heads=4 layers=4 seq=128 bs=64 epochs=10


In [3]:
def load_data(cfg):
    print("Loading WikiText-103...")
    ds = load_dataset('wikitext', 'wikitext-103-v1')
    tok = GPT2TokenizerFast.from_pretrained('gpt2')
    tok.pad_token = tok.eos_token

    def tokenize_and_chunk(split_name, max_rows=None):
        split = ds[split_name]
        if max_rows:
            split = split.select(range(min(max_rows, len(split))))

        # ── FIX: Concatenate all text, then chunk ──
        # Old code tokenized each row independently with padding → 80-90% wasted tokens.
        # Now: join all non-empty text → tokenize once → chunk into seq_len blocks.
        texts = [t for t in split['text'] if t.strip()]
        all_text = "\n".join(texts)
        all_ids = tok.encode(all_text)

        # Each chunk has seq_len+1 tokens (input = chunk[:-1], label = chunk[1:])
        sl = cfg.max_seq_len
        n = (len(all_ids) - 1) // sl
        chunks = [all_ids[i * sl : i * sl + sl + 1] for i in range(n)]

        print(f"  {split_name}: {len(texts):,} paragraphs → "
              f"{len(all_ids):,} tokens → {n:,} sequences of {sl}")
        return chunks

    train_chunks = tokenize_and_chunk('train',      cfg.max_train)
    val_chunks   = tokenize_and_chunk('validation',  cfg.max_val)
    test_chunks  = tokenize_and_chunk('test',        cfg.max_test)
    return train_chunks, val_chunks, test_chunks, tok


class LMDataset(Dataset):
    """Each item is a contiguous chunk — 100% token utilization, zero padding."""
    def __init__(self, chunks):
        self.chunks = chunks
    def __len__(self):
        return len(self.chunks)
    def __getitem__(self, i):
        c = torch.tensor(self.chunks[i], dtype=torch.long)
        return {'input_ids': c[:-1], 'labels': c[1:]}   # next-token prediction


train_chunks, val_chunks, test_chunks, tokenizer = load_data(cfg)
train_ds, val_ds, test_ds = LMDataset(train_chunks), LMDataset(val_chunks), LMDataset(test_chunks)
lkw = dict(num_workers=cfg.workers, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  **lkw)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, **lkw)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, **lkw)
print(f"\nTrain: {len(train_ds):,} ({len(train_loader)} batches)  "
      f"Val: {len(val_ds):,}  Test: {len(test_ds):,}")
print("100% token utilization — zero padding waste.")

Loading WikiText-103...


README.md: 0.00B [00:00, ?B/s]

wikitext-103-v1/test-00000-of-00001.parq(…):   0%|          | 0.00/722k [00:00<?, ?B/s]

wikitext-103-v1/train-00000-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

wikitext-103-v1/train-00001-of-00002.par(…):   0%|          | 0.00/156M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (3229635 > 1024). Running this sequence through the model will result in indexing errors


  train: 32,435 paragraphs → 3,229,635 tokens → 25,231 sequences of 128
  validation: 2,461 paragraphs → 247,137 tokens → 1,930 sequences of 128
  test: 2,891 paragraphs → 282,300 tokens → 2,205 sequences of 128

Train: 25,231 (395 batches)  Val: 1,930  Test: 2,205
100% token utilization — zero padding waste.


In [4]:
_EPS = 1e-7

def alpha_m(V):
    x = V + 40.0
    return torch.where(x.abs() < 1e-4, torch.ones_like(V),
           0.1 * x / (1.0 - torch.exp((-x / 10.0).clamp(-20, 20)) + _EPS))

def beta_m(V):
    return 4.0 * torch.exp((-(V + 65.0) / 18.0).clamp(-20, 20))

def alpha_h(V):
    return 0.07 * torch.exp((-(V + 65.0) / 20.0).clamp(-20, 20))

def beta_h(V):
    return 1.0 / (1.0 + torch.exp((-(V + 35.0) / 10.0).clamp(-20, 20)))

def alpha_n(V):
    x = V + 55.0
    return torch.where(x.abs() < 1e-4, torch.full_like(V, 0.1),
           0.01 * x / (1.0 - torch.exp((-x / 10.0).clamp(-20, 20)) + _EPS))

def beta_n(V):
    return 0.125 * torch.exp((-(V + 65.0) / 80.0).clamp(-20, 20))

with torch.no_grad():
    _Vr = torch.tensor(-65.0)
    _m0 = (alpha_m(_Vr) / (alpha_m(_Vr) + beta_m(_Vr))).item()
    _h0 = (alpha_h(_Vr) / (alpha_h(_Vr) + beta_h(_Vr))).item()
    _n0 = (alpha_n(_Vr) / (alpha_n(_Vr) + beta_n(_Vr))).item()
print(f"Resting gating: m={_m0:.4f}  h={_h0:.4f}  n={_n0:.4f}")

Resting gating: m=0.0529  h=0.5961  n=0.3177


In [5]:
class SurrogateSpike(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return (x > 0).float()
    @staticmethod
    def backward(ctx, g):
        x, = ctx.saved_tensors
        return g / (1.0 + 5.0 * x.abs()) ** 2

spike_fn = SurrogateSpike.apply


class QuantumHHNeuron(nn.Module):
    """
    Quantum HH neuron with Lindblad gating.
    All dynamics forced to fp32 (bf16 overflows on HH exponentials).
    Outputs normalized membrane voltage V in [-1, 1].
    """
    def __init__(self, d):
        super().__init__()
        self.d = d
        self.dt_sub = cfg.dt / cfg.n_substeps

        self.fc = nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)
        self.log_I = nn.Parameter(torch.tensor(math.log(20.0)))

        self.log_gNa = nn.Parameter(torch.full((d,), math.log(120.0)))
        self.log_gK  = nn.Parameter(torch.full((d,), math.log(36.0)))
        self.log_gL  = nn.Parameter(torch.full((d,), math.log(0.3)))
        self.register_buffer('ENa', torch.tensor(50.0))
        self.register_buffer('EK',  torch.tensor(-77.0))
        self.register_buffer('EL',  torch.tensor(-54.4))

        self.omega_raw = nn.Parameter(torch.full((3,), cfg.omega_init))
        self.eps_0     = nn.Parameter(torch.full((3,), cfg.epsilon_init))
        self.eps_1     = nn.Parameter(torch.full((3,), 0.01))
        self.log_gamma = nn.Parameter(torch.tensor(math.log(cfg.gamma_deph_init)))

    def _lindblad(self, V, p, cr, ci, a_fn, b_fn, gi, dt):
        a, b = a_fn(V), b_fn(V)
        om = F.softplus(self.omega_raw[gi])
        ep = self.eps_0[gi] + self.eps_1[gi] * (V + 65.0) / 100.0
        ga = self.log_gamma.exp()
        gc = (a + b) / 2.0 + 2.0 * ga

        p_inf = a / (a + b + _EPS)
        ed = torch.exp((-dt * (a + b)).clamp(min=-20.0))
        p2 = p_inf + (p - p_inf) * ed + dt * 2.0 * om * ci

        cd = torch.exp((-dt * gc).clamp(min=-20.0))
        cr2 = (cr + dt * 2.0 * ep * ci) * cd
        ci2 = (ci + dt * (-2.0 * ep * cr + om * (1.0 - 2.0 * p))) * cd

        p2 = p2.clamp(0.0, 1.0)
        mc = torch.sqrt(p2 * (1.0 - p2) + _EPS)
        cn = torch.sqrt(cr2**2 + ci2**2 + _EPS)
        s = torch.clamp(mc / cn, max=1.0)
        return p2, cr2 * s, ci2 * s

    def forward(self, x):
        """x: [B, T, d] -> V_norm [B, T, d], spike_rate (detached), coherence (detached)"""
        with torch.amp.autocast('cuda', enabled=False):
            x = x.float()
            B, T, D = x.shape

            V  = torch.full((B, D), cfg.v_reset, device=x.device)
            pm = torch.full((B, D), _m0, device=x.device)
            crm = torch.zeros(B, D, device=x.device)
            cim = torch.zeros(B, D, device=x.device)
            ph = torch.full((B, D), _h0, device=x.device)
            crh = torch.zeros(B, D, device=x.device)
            cih = torch.zeros(B, D, device=x.device)
            pn = torch.full((B, D), _n0, device=x.device)
            crn = torch.zeros(B, D, device=x.device)
            cin = torch.zeros(B, D, device=x.device)

            Is = self.log_I.exp()
            gNa, gK, gL = self.log_gNa.exp(), self.log_gK.exp(), self.log_gL.exp()
            dt = self.dt_sub

            voltages = []
            n_sp = 0
            t_coh = 0.0

            for t in range(T):
                raw = self.norm(self.fc(x[:, t]))
                I = Is * torch.tanh(raw / Is)

                for _ in range(cfg.n_substeps):
                    pm, crm, cim = self._lindblad(V, pm, crm, cim, alpha_m, beta_m, 0, dt)
                    ph, crh, cih = self._lindblad(V, ph, crh, cih, alpha_h, beta_h, 1, dt)
                    pn, crn, cin = self._lindblad(V, pn, crn, cin, alpha_n, beta_n, 2, dt)

                    I_Na = gNa * (pm**3) * ph * (V - self.ENa)
                    I_K  = gK  * (pn**4) * (V - self.EK)
                    I_L  = gL  * (V - self.EL)
                    V = (V + dt * (I - I_Na - I_K - I_L)).clamp(-100.0, 60.0)

                spike = spike_fn(V - cfg.v_thresh)
                V = torch.where(spike > 0.5, torch.full_like(V, cfg.v_reset), V)
                n_sp += spike.sum()

                coh = (torch.sqrt(crm**2 + cim**2 + _EPS).mean()
                     + torch.sqrt(crh**2 + cih**2 + _EPS).mean()
                     + torch.sqrt(crn**2 + cin**2 + _EPS).mean()) / 3.0
                t_coh += coh

                # Normalized voltage output [-1, 1]
                voltages.append((V + 100.0) / 160.0 * 2.0 - 1.0)

            out = torch.stack(voltages, dim=1)
            # Detach diagnostics — they should NOT create gradient paths
            sr = (n_sp / (B * T * D)).detach()
            mc = torch.nan_to_num(t_coh / T, nan=0.0).detach()
            return out, sr, mc


class QHHBlock(nn.Module):
    """
    QHH block with zero-initialized output projection.

    At init: proj output = 0 → forward returns exactly x.
    x.detach() breaks gradient flow from loss through 128 HH steps to embedding.
    QHH params still learn (gradient flows loss → hh_out → proj → neuron params).
    """
    def __init__(self, d):
        super().__init__()
        self.neuron = QuantumHHNeuron(d)
        self.proj = nn.Sequential(
            nn.Linear(d, d * 2),
            nn.GELU(),
            nn.Linear(d * 2, d),
        )
        self.norm = nn.LayerNorm(d)
        self.drop = nn.Dropout(cfg.dropout)

        # ZERO INIT: output starts at exactly 0
        nn.init.zeros_(self.proj[-1].weight)
        nn.init.zeros_(self.proj[-1].bias)

    def forward(self, x):
        # x.detach(): QHH backward never reaches embedding
        # QHH params still get gradients (through hh_out → proj → neuron)
        V, sr, coh = self.neuron(x.detach())
        hh_out = self.drop(self.norm(self.proj(V)))
        hh_out = torch.nan_to_num(hh_out, nan=0.0)
        # ADDITIVE: x is never scaled or modified
        return x + hh_out, sr, coh

print("QHH block defined.")
print("  Zero-init output → model starts as exact transformer")
print("  x.detach() → embedding gets clean gradients only")
print("  sr/coh detached → no spike regularization gradient path")

QHH block defined.
  Zero-init output → model starts as exact transformer
  x.detach() → embedding gets clean gradients only
  sr/coh detached → no spike regularization gradient path


In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d, 2).float() * (-math.log(10000.0) / d))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class HybridLM(nn.Module):
    """
    use_qhh=True:  Embedding → QHH Block → Transformer → LM Head
    use_qhh=False: Embedding → Transformer → LM Head (baseline)
    """
    def __init__(self, cfg, use_qhh=True):
        super().__init__()
        self.use_qhh = use_qhh
        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_enc = PositionalEncoding(cfg.d_model, cfg.max_seq_len)
        self.embed_drop = nn.Dropout(cfg.dropout)

        if use_qhh:
            self.qhh = QHHBlock(cfg.d_model)

        enc = nn.TransformerEncoderLayer(
            d_model=cfg.d_model, nhead=cfg.n_heads,
            dim_feedforward=cfg.d_model * 4, dropout=cfg.dropout,
            activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=cfg.n_layers)
        self.out_norm = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size)
        self.lm_head.weight = self.embedding.weight
        nn.init.normal_(self.embedding.weight, std=0.02)

    def _mask(self, T, dev):
        return torch.triu(torch.ones(T, T, device=dev), diagonal=1).bool()

    def forward(self, input_ids):
        B, T = input_ids.shape
        x = self.embed_drop(self.pos_enc(self.embedding(input_ids)))
        sr = torch.tensor(0.0, device=x.device)
        coh = torch.tensor(0.0, device=x.device)
        if self.use_qhh:
            x, sr, coh = self.qhh(x)
        x = self.transformer(x, mask=self._mask(T, x.device))
        return self.lm_head(self.out_norm(x)), sr, coh


# ── Build both models ──
model_baseline = HybridLM(cfg, use_qhh=False).to(cfg.device)
model_hybrid   = HybridLM(cfg, use_qhh=True).to(cfg.device)

# Copy baseline weights into hybrid (shared starting point)
baseline_sd = model_baseline.state_dict()
hybrid_sd = model_hybrid.state_dict()
for k in baseline_sd:
    if k in hybrid_sd:
        hybrid_sd[k] = baseline_sd[k].clone()
model_hybrid.load_state_dict(hybrid_sd)

n_h = sum(p.numel() for p in model_hybrid.parameters())
n_b = sum(p.numel() for p in model_baseline.parameters())
print(f"QHH-Transformer: {n_h:,} params")
print(f"Transformer:     {n_b:,} params")
print(f"QHH overhead:    {n_h - n_b:,} ({(n_h - n_b) / n_b * 100:.1f}%)")

# Verify identical init (eval mode → no dropout randomness)
print("\nVerifying identical init (eval mode)...")
model_hybrid.eval(); model_baseline.eval()
_ids = torch.randint(0, cfg.vocab_size, (2, cfg.max_seq_len), device=cfg.device)
with torch.no_grad(), torch.amp.autocast('cuda', dtype=cfg.dtype):
    _lh, _sr, _coh = model_hybrid(_ids)
    _lb, _, _ = model_baseline(_ids)
    diff = (_lh - _lb).abs().max().item()
    print(f"  Max logit difference: {diff:.8f}")
    print(f"  spike_rate={_sr:.4f}  coherence={_coh:.6f}")
    if diff < 0.01:
        print("  PERFECT: Models are identical at init.")
    elif diff < 1.0:
        print("  OK: Small numerical difference (LayerNorm fp rounding).")
    else:
        print(f"  WARNING: Difference {diff:.4f} — check QHH zero-init!")
model_hybrid.train(); model_baseline.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


QHH-Transformer: 16,406,108 params
Transformer:     16,075,601 params
QHH overhead:    330,507 (2.1%)

Verifying identical init (eval mode)...
  Max logit difference: 0.00000000
  spike_rate=0.2421  coherence=0.153820
  PERFECT: Models are identical at init.


HybridLM(
  (embedding): Embedding(50257, 256)
  (pos_enc): PositionalEncoding()
  (embed_drop): Dropout(p=0.1, inplace=False)
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (out_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (lm_head): Linear(in_features=256, out_features=50257, bias=True)
)

In [7]:
class Trainer:
    def __init__(self, model, cfg, name="model"):
        self.model = model
        self.cfg = cfg
        self.name = name
        self.has_qhh = hasattr(model, 'qhh') and model.use_qhh

        # Separate parameter groups for independent gradient clipping
        if self.has_qhh:
            qhh_params = list(model.qhh.parameters())
            qhh_ids = {id(p) for p in qhh_params}
            other_params = [p for p in model.parameters() if id(p) not in qhh_ids]
            self.opt = torch.optim.AdamW([
                {'params': other_params, 'lr': cfg.lr},
                {'params': qhh_params,   'lr': cfg.lr * 0.5},  # Lower LR for QHH
            ], weight_decay=cfg.weight_decay, betas=(0.9, 0.98), fused=True)
        else:
            self.opt = torch.optim.AdamW(
                model.parameters(), lr=cfg.lr,
                weight_decay=cfg.weight_decay, betas=(0.9, 0.98), fused=True)

        total_steps = len(train_loader) * cfg.epochs
        self.sched = torch.optim.lr_scheduler.OneCycleLR(
            self.opt, max_lr=cfg.lr, total_steps=total_steps,
            pct_start=cfg.warmup_frac, anneal_strategy='cos')

        self.scaler = torch.amp.GradScaler('cuda', enabled=(cfg.dtype == torch.float16))
        self.best_loss = float('inf')
        self.wait = 0
        self.nan_count = 0
        self.history = {k: [] for k in
            ['train_loss','train_ppl','val_loss','val_ppl',
             'spike_rate','coherence','lr','time']}

    def _ppl(self, l, t):
        return math.exp(min(l / max(t, 1), 100)) if t else float('inf')

    def _run(self, loader, train=False, desc=""):
        self.model.train(train)
        tl = tt = nb_ = nan_b = 0
        tsr = tco = 0.0
        if train:
            self.opt.zero_grad()

        pbar = tqdm(loader, desc=desc)
        for i, batch in enumerate(pbar):
            ids = batch['input_ids'].to(cfg.device, non_blocking=True)
            lab = batch['labels'].to(cfg.device, non_blocking=True)

            with torch.amp.autocast('cuda', dtype=cfg.dtype):
                logits, sr, coh = self.model(ids)
                loss_s = F.cross_entropy(
                    logits.view(-1, cfg.vocab_size),
                    lab.view(-1), ignore_index=-100, reduction='sum')
                nt = (lab != -100).sum().item()

            # NaN guard — skip entire batch before backward
            if torch.isnan(loss_s) or torch.isinf(loss_s) or nt == 0:
                nan_b += 1
                if train:
                    self.opt.zero_grad()
                continue

            if train:
                loss_mean = loss_s / nt
                # sr is already detached (from neuron), so no spike_reg gradient path
                self.scaler.scale(loss_mean).backward()

                self.scaler.unscale_(self.opt)

                # NaN gradient guard
                bad_grad = any(
                    p.grad is not None and (torch.isnan(p.grad).any() or torch.isinf(p.grad).any())
                    for p in self.model.parameters())
                if bad_grad:
                    self.opt.zero_grad()
                    nan_b += 1
                    continue

                # ═══════════════════════════════════════════
                # SEPARATE gradient clipping — the key fix.
                # QHH 128-step backward produces large gradients.
                # If clipped jointly, these inflate the global norm
                # and suppress clean transformer gradients to near-zero.
                # ═══════════════════════════════════════════
                if self.has_qhh:
                    # Clip transformer params
                    transformer_params = [p for n, p in self.model.named_parameters()
                                          if 'qhh' not in n and p.grad is not None]
                    if transformer_params:
                        torch.nn.utils.clip_grad_norm_(transformer_params, cfg.grad_clip)
                    # Clip QHH params separately (more lenient)
                    qhh_params = [p for n, p in self.model.named_parameters()
                                  if 'qhh' in n and p.grad is not None]
                    if qhh_params:
                        torch.nn.utils.clip_grad_norm_(qhh_params, cfg.qhh_grad_clip)
                else:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.grad_clip)

                self.scaler.step(self.opt)
                self.scaler.update()
                self.opt.zero_grad()
                self.sched.step()

            tl += loss_s.item()
            tt += nt
            tsr += (sr.item() if isinstance(sr, torch.Tensor) else sr)
            tco += (coh.item() if isinstance(coh, torch.Tensor) else coh)
            nb_ += 1

            if train and i % 20 == 0:
                pbar.set_postfix(
                    ppl=f'{self._ppl(tl, tt):.1f}',
                    sr=f'{tsr / max(nb_, 1):.4f}',
                    nan=f'{nan_b}')

        self.nan_count += nan_b
        return {'loss': tl / max(tt, 1), 'ppl': self._ppl(tl, tt),
                'spike_rate': tsr / max(nb_, 1), 'coherence': tco / max(nb_, 1)}

    def train(self):
        print(f"\n{'='*70}\nTRAINING: {self.name}\n{'='*70}")
        for ep in range(cfg.epochs):
            t0 = time.time()
            tr = self._run(train_loader, True,
                           f"[{self.name}] Train {ep+1}/{cfg.epochs}")
            va = self._run(val_loader, False,
                           f"[{self.name}] Val {ep+1}/{cfg.epochs}")
            el = time.time() - t0

            for k, v in [('train_loss',tr['loss']),('train_ppl',tr['ppl']),
                ('val_loss',va['loss']),('val_ppl',va['ppl']),
                ('spike_rate',tr['spike_rate']),('coherence',tr['coherence']),
                ('lr',self.sched.get_last_lr()[0]),('time',el)]:
                self.history[k].append(v)

            print(f"\nEp {ep+1}/{cfg.epochs} ({el:.0f}s)  "
                  f"Train PPL: {tr['ppl']:.2f}  Val PPL: {va['ppl']:.2f}  "
                  f"sr={tr['spike_rate']:.4f} coh={tr['coherence']:.6f}")
            if self.nan_count > 0:
                print(f"  NaN batches total: {self.nan_count}")

            if va['loss'] < self.best_loss:
                self.best_loss = va['loss']
                self.wait = 0
                self._save('best.pt', ep, va)
                print(f"  >> Best val PPL: {va['ppl']:.2f}")
            else:
                self.wait += 1
                print(f"  No improvement ({self.wait}/{cfg.patience})")
            if (ep + 1) % 5 == 0:
                self._save(f'ep{ep+1}.pt', ep, va)
            if self.wait >= cfg.patience:
                print("  Early stop.")
                break

        print(f"[{self.name}] Best: {math.exp(min(self.best_loss, 100)):.2f}")

    def _save(self, name, ep, m):
        p = cfg.save_dir / f'{self.name}_{name}'
        torch.save({'epoch': ep, 'model_state_dict': self.model.state_dict(),
                     'metrics': m, 'history': self.history}, p)
        print(f"    -> {p}")

print("Trainer ready.")
print("  Separate grad clipping: transformer and QHH clipped independently.")
print(f"  Transformer clip: {cfg.grad_clip}  QHH clip: {cfg.qhh_grad_clip}")

Trainer ready.
  Separate grad clipping: transformer and QHH clipped independently.
  Transformer clip: 1.0  QHH clip: 5.0


In [ ]:
# Train hybrid, then baseline
trainer_h = Trainer(model_hybrid, cfg, "qhh-transformer")
trainer_h.train()

torch.cuda.empty_cache()

trainer_b = Trainer(model_baseline, cfg, "transformer")
trainer_b.train()


TRAINING: qhh-transformer


[qhh-transformer] Train 1/10:   0%|          | 0/395 [00:00<?, ?it/s]

[qhh-transformer] Val 1/10:   0%|          | 0/31 [00:00<?, ?it/s]


Ep 1/10 (538s)  Train PPL: 2681.78  Val PPL: 1650.75  sr=0.2386 coh=0.151028
    -> qhh_hybrid_quick/qhh-transformer_best.pt
  >> Best val PPL: 1650.75


[qhh-transformer] Train 2/10:   0%|          | 0/395 [00:00<?, ?it/s]

...


Ep 3/10 (540s)  Train PPL: 862.67  Val PPL: 645.09  sr=0.2375 coh=0.164738
    -> qhh_hybrid_quick/qhh-transformer_best.pt
  >> Best val PPL: 645.09


[qhh-transformer] Train 4/10:   0%|          | 0/395 [00:00<?, ?it/s]

[qhh-transformer] Val 4/10:   0%|          | 0/31 [00:00<?, ?it/s]


Ep 4/10 (545s)  Train PPL: 612.62  Val PPL: 494.02  sr=0.2352 coh=0.164376
    -> qhh_hybrid_quick/qhh-transformer_best.pt
  >> Best val PPL: 494.02


[qhh-transformer] Train 5/10:   0%|          | 0/395 [00:00<?, ?it/s]

In [ ]:
# Load best + evaluate
def load_best(model, name):
    p = cfg.save_dir / f'{name}_best.pt'
    ck = torch.load(p, map_location=cfg.device, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    return ck

ck_h = load_best(model_hybrid, 'qhh-transformer')
ck_b = load_best(model_baseline, 'transformer')
h_h = ck_h['history']; h_b = ck_b['history']

ev_h = Trainer(model_hybrid, cfg, "qhh-transformer")
ev_b = Trainer(model_baseline, cfg, "transformer")
test_h = ev_h._run(test_loader, False, "QHH Test")
test_b = ev_b._run(test_loader, False, "Trans Test")

print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)
print(f"{'Model':<25} {'PPL':>8} {'Loss':>8} {'Spike':>8} {'Coh':>10}")
print("-" * 65)
print(f"{'QHH-Transformer':<25} {test_h['ppl']:>8.2f} {test_h['loss']:>8.4f} "
      f"{test_h['spike_rate']:>8.4f} {test_h['coherence']:>10.6f}")
print(f"{'Transformer':<25} {test_b['ppl']:>8.2f} {test_b['loss']:>8.4f} "
      f"{'N/A':>8} {'N/A':>10}")
d = (test_b['ppl'] - test_h['ppl']) / test_b['ppl'] * 100
print(f"\nImprovement: {d:+.2f}% perplexity")

# QHH contribution
with torch.no_grad():
    model_hybrid.eval()
    _ids = torch.randint(0, cfg.vocab_size, (2, cfg.max_seq_len), device=cfg.device)
    with torch.amp.autocast('cuda', dtype=cfg.dtype):
        x = model_hybrid.embed_drop(model_hybrid.pos_enc(model_hybrid.embedding(_ids)))
        x_after, _, _ = model_hybrid.qhh(x)
        delta = (x_after - x).abs().mean().item()
    print(f"\nQHH contribution magnitude: {delta:.6f}")
    print("  " + ("Active contribution." if delta > 0.001 else "Negligible."))

# Quantum params
n = model_hybrid.qhh.neuron
om = F.softplus(n.omega_raw).detach().cpu().tolist()
ga = n.log_gamma.exp().item()
r = ga / (np.mean(om) + 1e-8)
print(f"\nOmega={[f'{o:.3f}' for o in om]} gamma={ga:.4f} ratio={r:.2f}", end="  ")
print("quantum-coherent" if r < 1 else "weak-quantum" if r < 10 else "near-classical")

print(f"\nAvg epoch: hybrid={np.mean(h_h['time']):.0f}s  "
      f"baseline={np.mean(h_b['time']):.0f}s  "
      f"overhead={(np.mean(h_h['time'])-np.mean(h_b['time']))/np.mean(h_b['time'])*100:+.1f}%")
print("=" * 70)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
Ch, Cb = '#E91E63', '#2196F3'

for ax, kh, kb, yl, t in [
    (axes[0,0], 'val_ppl', 'val_ppl', 'PPL', 'Val Perplexity'),
    (axes[0,1], 'train_ppl', 'train_ppl', 'PPL', 'Train Perplexity'),
    (axes[0,2], 'val_loss', 'val_loss', 'Loss', 'Val Loss')]:
    ax.plot(h_h[kh], label='QHH-Trans', color=Ch, lw=2)
    ax.plot(h_b[kb], label='Trans', color=Cb, lw=2, ls='--')
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl)
    ax.set_title(t, fontweight='bold'); ax.legend(); ax.grid(True, alpha=0.3)

axes[1,0].plot(h_h['spike_rate'], color='green', lw=2)
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Spike Rate')
axes[1,0].set_title('Spike Activity', fontweight='bold'); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(h_h['coherence'], color='purple', lw=2)
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('|coherence|')
axes[1,1].set_title('Quantum Coherence', fontweight='bold'); axes[1,1].grid(True, alpha=0.3)

w = 0.35; x = np.arange(max(len(h_h['time']), len(h_b['time'])))
axes[1,2].bar(x[:len(h_h['time'])]-w/2, h_h['time'], w, label='QHH-T', color=Ch, alpha=.8)
axes[1,2].bar(x[:len(h_b['time'])]+w/2, h_b['time'], w, label='Trans', color=Cb, alpha=.8)
axes[1,2].set_xlabel('Epoch'); axes[1,2].set_ylabel('Time (s)')
axes[1,2].set_title('Wall-Clock', fontweight='bold')
axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3, axis='y')

plt.suptitle(f'QHH-Transformer vs Transformer — {MODE}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(cfg.save_dir / 'comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"Saved: {cfg.save_dir / 'comparison.png'}")

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new=80, temperature=0.8, top_p=0.95):
    model.eval()
    ids = tokenizer.encode(prompt, return_tensors='pt').to(cfg.device)
    for _ in range(max_new):
        if ids.size(1) > cfg.max_seq_len:
            ids = ids[:, -cfg.max_seq_len:]
        with torch.amp.autocast('cuda', dtype=cfg.dtype):
            logits, _, _ = model(ids)
        nl = logits[:, -1, :] / temperature
        sl, si = torch.sort(nl, descending=True)
        cum = torch.cumsum(F.softmax(sl, dim=-1), dim=-1)
        sl[cum - F.softmax(sl, dim=-1) >= top_p] = float('-inf')
        tok = si.gather(-1, torch.multinomial(F.softmax(sl, dim=-1), 1))
        ids = torch.cat([ids, tok], dim=1)
        if tok.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(ids[0], skip_special_tokens=True)

for label, m in [("QHH-Transformer", model_hybrid), ("Transformer", model_baseline)]:
    print(f"\n{'='*70}\n{label}\n{'='*70}")
    for p in ["The history of artificial intelligence",
              "In a groundbreaking study, researchers found that",
              "The relationship between quantum mechanics and"]:
        print(f"\nPrompt: {p}\n" + "-" * 60)
        print(generate(m, p))

In [ ]:
results = {
    'mode': MODE, 'seed': SEED,
    'test': {'hybrid': test_h, 'baseline': test_b},
    'params': {'hybrid': int(n_h), 'baseline': int(n_b)},
    'histories': {'hybrid': h_h, 'baseline': h_b},
    'quantum_params': {
        'omega': F.softplus(model_hybrid.qhh.neuron.omega_raw).detach().cpu().tolist(),
        'gamma_deph': model_hybrid.qhh.neuron.log_gamma.exp().item(),
    },
    'config': {k: str(v) if isinstance(v, (Path, torch.device, torch.dtype)) else v
               for k, v in vars(cfg).items()},
}
with open(cfg.save_dir / 'results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"Saved to {cfg.save_dir}/")
for p in sorted(cfg.save_dir.iterdir()):
    print(f"  {p.name}")
print(f"\nDone. Change MODE to 'serious' or 'neurips' and re-run.")